# Deep Researcher

In [ ]:
!pip install -q langgraph langchain-openai langchain-community serpapi beautifulsoup4 lxml pydantic python-dotenv gradio requests sendgrid

## Imports

In [ ]:
from typing import Annotated, TypedDict, List, Dict, Any, Optional
import json
import os
import uuid

import gradio as gr
import requests
import sendgrid
import serpapi
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import Image, display
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from pydantic import BaseModel, Field
from sendgrid.helpers.mail import Email, Mail, Content, To

## Environment Variables

In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
serp_api_key = os.getenv("SERP_API_KEY")
sendgrid_api_key = os.getenv("SENDGRID_API_KEY")
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
sender_email = os.getenv("EMAIL_SENDER")
receiver_email = os.getenv("EMAIL_RECEIVER")
langsmith_api_key = os.getenv("LANGSMITH_API_KEY")

for name, val in [
    ("OPENAI_API_KEY", openai_api_key),
    ("SERP_API_KEY", serp_api_key),
    ("SENDGRID_API_KEY", sendgrid_api_key),
    ("PUSHOVER_USER", pushover_user),
    ("PUSHOVER_TOKEN", pushover_token),
    ("EMAIL_SENDER", sender_email),
    ("EMAIL_RECEIVER", receiver_email),
]:
    if not val:
        raise ValueError(f"{name} is not set")

if langsmith_api_key:
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = langsmith_api_key
    os.environ.setdefault("LANGSMITH_PROJECT", "ike-deep-researcher")
    os.environ.setdefault("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")

AGENT_MODEL = "gpt-4o-mini"

## LLM Instances

In [ ]:
guardrail_llm = ChatOpenAI(model=AGENT_MODEL, temperature=0, max_tokens=500)
clarifier_llm = ChatOpenAI(model=AGENT_MODEL, temperature=0.5, max_tokens=1000)
worker_llm = ChatOpenAI(model=AGENT_MODEL, temperature=0.7, max_tokens=4000)
evaluator_llm = ChatOpenAI(model=AGENT_MODEL, temperature=0.3, max_tokens=1000)
notifier_llm = ChatOpenAI(model=AGENT_MODEL, temperature=0.5, max_tokens=3000)

## Structured Outputs (Pydantic Models)

In [ ]:
class GuardrailResult(BaseModel):
    is_appropriate: bool = Field(description="Whether the research topic is appropriate and safe to process")
    reason: str = Field(description="Brief explanation of why the topic was approved or rejected")
    category: str = Field(description="Category of the issue if inappropriate, or 'approved' if appropriate")


class ClarifierOutput(BaseModel):
    question: Optional[str] = Field(default=None, description="A single clarifying question to ask the user, or None if ready")
    ready_to_research: bool = Field(description="True if the query is clear enough to proceed with research")
    refined_query: Optional[str] = Field(default=None, description="Refined version of the user's request once clarification is complete")


class EvaluatorOutput(BaseModel):
    feedback: str = Field(description="Feedback on the research output")
    meets_requirements: bool = Field(description="Whether the output meets the original search request")
    retry_recommended: bool = Field(description="Whether a retry would improve the output")


class FormattedEmail(BaseModel):
    subject: str = Field(description="Clear, concise email subject line")
    html_body: str = Field(
        description="HTML body using <h2>, <p>, <ul> tags. No letter salutations or placeholders."
    )

## Graph State

In [ ]:
class State(TypedDict, total=False):
    messages: Annotated[List[Any], add_messages]
    guard_ok: bool
    clarification_count: int           # questions asked so far (max 3)
    clarification_complete: bool
    original_query: str                # preserved original user query
    refined_query: str                 # query after clarification
    retry_count: int                   # evaluator retries (max 3)
    evaluator_feedback: Optional[str]
    success_criteria_met: bool
    last_report: Optional[str]

## Tools

In [ ]:
@tool
def google_search(query: str) -> str:
    """Search Google via SerpAPI."""
    try:
        client = serpapi.Client(api_key=serp_api_key)
        response = client.search(engine="google", q=query)
        results = response.get("organic_results") or []
        if not results:
            return "No results found."
        out = []
        for r in results[:5]:
            out.append(f"- {r.get('title', '')}: {r.get('snippet', '')}\n  URL: {r.get('link', '')}")
        return "\n".join(out)
    except Exception as e:
        return f"Search error: {e}"


@tool
def fetch_web_page(url: str) -> str:
    """Fetch a URL and return plain text content for analysis."""
    try:
        r = requests.get(
            url,
            timeout=20,
            headers={"User-Agent": "DeepResearcher/1.0 (research assistant)"},
        )
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "lxml")
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()
        text = soup.get_text(separator="\n", strip=True)
        return text[:12000] if len(text) > 12000 else text
    except Exception as e:
        return f"Failed to fetch URL: {e}"


wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())


@tool
def send_push_notification(message: str) -> str:
    """Send a push notification via Pushover."""
    try:
        url = "https://api.pushover.net/1/messages.json"
        payload = {"token": pushover_token, "user": pushover_user, "message": message}
        r = requests.post(url, payload, timeout=30)
        r.raise_for_status()
        return "Notification sent."
    except Exception as e:
        return f"Notification failed: {e}"


@tool
def send_email_tool(subject: str, html_body: str) -> str:
    """Send an HTML email via SendGrid."""
    try:
        sg = sendgrid.SendGridAPIClient(api_key=sendgrid_api_key)
        from_email = Email(sender_email)
        to_email = To(receiver_email)
        content = Content("text/html", html_body)
        mail = Mail(from_email, to_email, subject, content).get()
        response = sg.client.mail.send.post(request_body=mail)
        return f"Email sent (status {response.status_code})."
    except Exception as e:
        return f"Email failed: {e}"

## Tool Bindings

In [ ]:
worker_tools = [google_search, wiki_tool, fetch_web_page]
worker_tool_node = ToolNode(worker_tools)
worker_llm_with_tools = worker_llm.bind_tools(worker_tools)

guardrail_llm_structured = guardrail_llm.with_structured_output(GuardrailResult)
clarifier_llm_structured = clarifier_llm.with_structured_output(ClarifierOutput)
evaluator_llm_structured = evaluator_llm.with_structured_output(EvaluatorOutput)
email_format_llm = notifier_llm.with_structured_output(FormattedEmail)

## Graph Nodes

### Guardrail Node
LLM-based content moderation — blocks inappropriate research topics.

In [ ]:
GUARDRAIL_SYSTEM = """You are a content moderation agent responsible for reviewing research topics before processing.

Your task is to determine if a research query is appropriate and safe to process.

A research topic is INAPPROPRIATE if it involves:
- Explicit sexual content or pornography
- Illegal activities (drugs, weapons trafficking, fraud, hacking, etc.)
- Violence, harm, or dangerous instructions
- Hate speech, discrimination, or harassment
- Child exploitation or endangerment
- Self-harm or suicide-related content
- Privacy violations or doxxing
- Misinformation intended to cause harm

A research topic is APPROPRIATE if it involves:
- Academic or educational research
- Medical, scientific, or technical topics
- Business, economics, or policy analysis
- Historical or cultural studies
- General information seeking
- Legitimate security research or safety topics

Analyze the query carefully and make a fair judgment. Err on the side of allowing legitimate research while blocking clearly inappropriate content."""


def guardrail_node(state: State) -> Dict[str, Any]:
    """Check the latest user message for appropriateness."""
    text = ""
    for m in reversed(state["messages"]):
        if isinstance(m, HumanMessage):
            text = str(m.content)
            break

    if not text.strip():
        return {
            "guard_ok": False,
            "messages": [AIMessage(content="Please provide a research query.")],
        }

    if len(text.strip()) < 10:
        return {
            "guard_ok": False,
            "messages": [AIMessage(content="Your query is too short. Please provide at least 10 characters.")],
        }

    result: GuardrailResult = guardrail_llm_structured.invoke(
        [SystemMessage(content=GUARDRAIL_SYSTEM), HumanMessage(content=f"Research topic: {text}")]
    )

    if result.is_appropriate:
        updates: Dict[str, Any] = {"guard_ok": True}
        if not state.get("original_query"):
            updates["original_query"] = text
        return updates
    else:
        return {
            "guard_ok": False,
            "messages": [
                AIMessage(
                    content=f"**Research topic blocked**: {result.reason}\n\n"
                    f"**Category**: {result.category}\n\n"
                    "Please provide a different, appropriate research topic."
                )
            ],
        }


def route_after_guard(state: State) -> str:
    return "clarifier" if state.get("guard_ok") else END

### Clarifier Node
Asks the user ONE clarifying question at a time (max 3). The graph stops after each question so the user can reply in the chat.

In [ ]:
CLARIFIER_SYSTEM = """You are a research clarifying agent. Given a user's research query and any conversation history, decide:

1. If the query is clear and specific enough to proceed with research, set ready_to_research=True and provide a refined_query that captures all details.
2. If you need more info, set ready_to_research=False and provide exactly ONE targeted clarifying question.

Your questions should:
- Narrow the scope of the research topic
- Identify the user's goal (learning, decision-making, comparison, etc.)
- Clarify constraints (timeframe, geography, depth, etc.)
- Determine desired output format (summary, report, data, etc.)

Guidelines:
- Ask only ONE question at a time
- Prioritize high-impact questions that significantly change the research direction
- Be specific, not generic
- If the user has already provided answers in the conversation, factor those in
- If the query is already detailed and specific, just proceed (ready_to_research=True)"""

MAX_CLARIFICATIONS = 3


def _format_conversation(messages: List[Any]) -> str:
    """Format message history for LLM consumption."""
    lines: List[str] = []
    for m in messages:
        if isinstance(m, HumanMessage):
            lines.append(f"User: {m.content}")
        elif isinstance(m, AIMessage):
            lines.append(f"Assistant: {m.content}")
        elif isinstance(m, SystemMessage):
            lines.append(f"System: {m.content}")
        elif isinstance(m, ToolMessage):
            lines.append(f"Tool ({m.name}): {m.content[:200]}")
    return "\n".join(lines)


def clarifier_node(state: State) -> Dict[str, Any]:
    """Ask one clarifying question or declare ready to research."""
    count = state.get("clarification_count", 0)

    convo = _format_conversation(state["messages"])

    # If we've asked max questions, force ready
    if count >= MAX_CLARIFICATIONS:
        forced: ClarifierOutput = clarifier_llm_structured.invoke([
            SystemMessage(
                content=CLARIFIER_SYSTEM
                + "\n\nYou have already asked the maximum number of questions. "
                "You MUST set ready_to_research=True and provide the refined_query."
            ),
            HumanMessage(content=f"Conversation so far:\n{convo}"),
        ])
        refined = forced.refined_query or state.get("original_query", "")
        return {
            "clarification_complete": True,
            "refined_query": refined,
            "messages": [AIMessage(content=f"Great, I have enough context. Starting research on: {refined}")],
        }

    result: ClarifierOutput = clarifier_llm_structured.invoke([
        SystemMessage(content=CLARIFIER_SYSTEM),
        HumanMessage(content=f"Conversation so far:\n{convo}"),
    ])

    if result.ready_to_research:
        refined = result.refined_query or state.get("original_query", "")
        return {
            "clarification_complete": True,
            "refined_query": refined,
            "messages": [AIMessage(content=f"Great, I have enough context. Starting research on: {refined}")],
        }
    else:
        question = result.question or "Could you provide more details about your research needs?"
        return {
            "clarification_count": count + 1,
            "clarification_complete": False,
            "messages": [AIMessage(content=question)],
        }


def route_after_clarifier(state: State) -> str:
    if state.get("clarification_complete"):
        return "worker"
    return END  # wait for user reply

### Worker Node
Research LLM with access to Google Search, Wikipedia tools.

In [ ]:
def worker_node(state: State) -> Dict[str, Any]:
    """Perform research using tools, producing a detailed report."""
    refined = state.get("refined_query") or state.get("original_query", "")
    system = f"""You are a thorough research worker. Your task is to research the following query and produce a detailed, well-structured report.

Research query: {refined}

You have these tools available:
- google_search: Search Google for relevant results
- wikipedia: Search Wikipedia for background information
- fetch_web_page: Fetch and read a specific URL

Strategy:
1. Start with a Google search to find relevant sources
2. Use Wikipedia for background/context
3. Fetch specific web pages for detailed information
4. Synthesize all findings into a comprehensive report

When you have enough evidence, write the FINAL report as markdown in your last message (no tool calls).
The report should include:
- A clear title
- Organized sections (summary, key findings, supporting detail)
- Citations or mentions of sources used
- Include any relevant nuances, caveats, or limitations in the evidence

Aim for a thorough, detailed report (3-5 pages, ~800-1000 words)."""

    if state.get("evaluator_feedback"):
        system += f"""\n\nYour previous attempt was rejected by the evaluator. Address this feedback:\n{state['evaluator_feedback']}"""

    messages = [SystemMessage(content=system)] + list(state["messages"])
    response = worker_llm_with_tools.invoke(messages)

    out: Dict[str, Any] = {"messages": [response]}
    if not getattr(response, "tool_calls", None):
        out["last_report"] = response.content or ""
    return out


def route_worker(state: State) -> str:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return "evaluator"

### Evaluator Node
Checks if the worker's output meets the original research request. Can request up to 3 retries.

In [ ]:
MAX_RETRIES = 3

EVALUATOR_SYSTEM = """You are a research quality evaluator. Given the original research request and the worker's output, determine:

1. Does the output adequately address the original research request?
2. Is it accurate, well-structured, and sufficiently detailed?
3. Should the worker retry with specific feedback?

Be fair but thorough. A good report should:
- Directly answer the research question
- Cite or reference sources
- Be well-organized with clear sections
- Provide enough detail to be useful

If the report is adequate, set meets_requirements=True.
If it needs improvement and a retry would help, set retry_recommended=True with specific feedback."""


def evaluator_node(state: State) -> Dict[str, Any]:
    """Evaluate the worker's research output."""
    report = state.get("last_report", "")
    original = state.get("original_query", "")
    refined = state.get("refined_query", original)
    retry_count = state.get("retry_count", 0)

    user_msg = f"""Original query: {original}
Refined query: {refined}
Retry count: {retry_count}/{MAX_RETRIES}

Worker's report:
{report[:8000]}"""

    if state.get("evaluator_feedback"):
        user_msg += f"\n\nPrevious feedback (avoid repeat issues): {state['evaluator_feedback']}"

    result: EvaluatorOutput = evaluator_llm_structured.invoke(
        [SystemMessage(content=EVALUATOR_SYSTEM), HumanMessage(content=user_msg)]
    )

    if result.meets_requirements:
        return {
            "success_criteria_met": True,
            "messages": [AIMessage(content=f"Evaluator: Approved. {result.feedback}")],
        }

    if result.retry_recommended and retry_count < MAX_RETRIES:
        return {
            "retry_count": retry_count + 1,
            "evaluator_feedback": result.feedback,
            "success_criteria_met": False,
            "messages": [
                AIMessage(
                    content=f"Evaluator: Retry requested ({retry_count + 1}/{MAX_RETRIES}). "
                    f"Feedback: {result.feedback}"
                )
            ],
        }

    # Max retries reached or no retry recommended — accept as-is
    return {
        "success_criteria_met": True,
        "messages": [AIMessage(content=f"Evaluator: Accepted (max retries reached). {result.feedback}")],
    }


def route_after_evaluator(state: State) -> str:
    if state.get("success_criteria_met"):
        return "notifier"
    return "worker"  # retry

### Notifier Node
Sends a push notification summary and the full research report via email.

In [ ]:
def notifier_node(state: State) -> Dict[str, Any]:
    """Send push notification (summary) and email (full report)."""
    report = (state.get("last_report") or "").strip() or "(empty report)"
    summary = (report[:450] + "...") if len(report) > 450 else report
    parts: List[str] = []

    # Push notification
    push_result = send_push_notification.invoke({"message": summary})
    parts.append(f"Push: {push_result}")

    # Format and send email
    try:
        fmt_system = (
            "You turn a research report into a compact HTML email. "
            "Subject: under 80 characters, specific to the topic. "
            "Body: use <h2>, <p>, <ul>/<li> where helpful; preserve key facts, sources, and structure. "
            "Do NOT use letter-style openings/closings or placeholders like [Recipient's Name]. "
            "Start directly with substantive content."
        )
        formatted: FormattedEmail = email_format_llm.invoke(
            [SystemMessage(content=fmt_system), HumanMessage(content=report[:20000])]
        )
        body = formatted.html_body
        for bad in ["Dear [Recipient's Name],", "Dear [Recipient's Name]", "[Your Name]", "[Recipient's Name]"]:
            body = body.replace(bad, "")

        email_result = send_email_tool.invoke({"subject": formatted.subject, "html_body": body.strip()})
        parts.append(f"Email: {email_result}")
    except Exception as e:
        parts.append(f"Email failed: {e}")

    notification_status = " | ".join(parts)
    return {
        "messages": [
            AIMessage(
                content=f"{report}\n\n---\n**Notifications:** {notification_status}"
            )
        ],
    }

## Build Graph

In [ ]:
memory = MemorySaver()

graph_builder = StateGraph(State)

# Add nodes
graph_builder.add_node("guardrail", guardrail_node)
graph_builder.add_node("clarifier", clarifier_node)
graph_builder.add_node("worker", worker_node)
graph_builder.add_node("tools", worker_tool_node)
graph_builder.add_node("evaluator", evaluator_node)
graph_builder.add_node("notifier", notifier_node)

# Edges
graph_builder.add_edge(START, "guardrail")
graph_builder.add_conditional_edges(
    "guardrail", route_after_guard, {"clarifier": "clarifier", END: END}
)
graph_builder.add_conditional_edges(
    "clarifier", route_after_clarifier, {"worker": "worker", END: END}
)
graph_builder.add_conditional_edges(
    "worker", route_worker, {"tools": "tools", "evaluator": "evaluator"}
)
graph_builder.add_edge("tools", "worker")
graph_builder.add_conditional_edges(
    "evaluator", route_after_evaluator, {"notifier": "notifier", "worker": "worker"}
)
graph_builder.add_edge("notifier", END)

graph = graph_builder.compile(checkpointer=memory)

### Display Graph

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

## Chat Interface

In [ ]:
def _chat_preview(messages: List[Any]) -> str:
    """Extract the latest assistant message for display."""
    for m in reversed(messages):
        if isinstance(m, AIMessage):
            c = (m.content or "").strip()
            if c:
                return c
    return "Loading..."


async def stream_chat(message: str, history: list, thread_id: str):
    """Stream graph execution, yielding updates as each node completes."""
    if not message or not str(message).strip():
        yield history, thread_id
        return

    msg = str(message).strip()
    hist = list(history)
    hist.append({"role": "user", "content": msg})
    hist.append({"role": "assistant", "content": "Running workflow..."})
    yield hist, thread_id

    config = {"configurable": {"thread_id": thread_id}}
    state_in = {"messages": [HumanMessage(content=msg)]}

    try:
        async for state in graph.astream(state_in, config, stream_mode="values"):
            msgs = state.get("messages") or []
            hist[-1] = {"role": "assistant", "content": _chat_preview(msgs)}
            yield hist, thread_id
    except Exception as e:
        hist[-1] = {"role": "assistant", "content": f"**Error:** {e}"}
        yield hist, thread_id


def reset_conversation():
    """Reset chat and generate a new thread ID."""
    return [], str(uuid.uuid4())

### Gradio UI

In [ ]:
with gr.Blocks(title="Deep Researcher") as ui:
    gr.Markdown(
        "# Deep Researcher\n"
    )
    thread_id = gr.State(lambda: str(uuid.uuid4()))
    chat = gr.Chatbot(height=520, label="Chat")
    box = gr.Textbox(
        placeholder="What would you like to research?",
        label="Your message",
        lines=2,
    )
    with gr.Row():
        go = gr.Button("Send", variant="primary")
        reset = gr.Button("Reset")

    go.click(stream_chat, [box, chat, thread_id], [chat, thread_id]).then(
        lambda: "", outputs=box
    )
    box.submit(stream_chat, [box, chat, thread_id], [chat, thread_id]).then(
        lambda: "", outputs=box
    )
    reset.click(reset_conversation, outputs=[chat, thread_id])

ui.launch(inbrowser=True)